## 1. Kaggle Setup

In [ ]:

from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle (2).json


## 2. MRI / CT Scan Dataset

In [ ]:

# Example keyword: medical MRI image dataset
# Replace dataset-name with actual Kaggle dataset slug

!kaggle datasets download -d navoneel/brain-mri-images-for-brain-tumor-detection
!unzip brain-mri-images-for-brain-tumor-detection.zip


Dataset URL: https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection
License(s): copyright-authors
  0% 0.00/15.1M [00:00<?, ?B/s]
100% 15.1M/15.1M [00:00<00:00, 924MB/s]
Archive:  brain-mri-images-for-brain-tumor-detection.zip
  inflating: brain_tumor_dataset/no/1 no.jpeg  
  inflating: brain_tumor_dataset/no/10 no.jpg  
  inflating: brain_tumor_dataset/no/11 no.jpg  
  inflating: brain_tumor_dataset/no/12 no.jpg  
  inflating: brain_tumor_dataset/no/13 no.jpg  
  inflating: brain_tumor_dataset/no/14 no.jpg  
  inflating: brain_tumor_dataset/no/15 no.jpg  
  inflating: brain_tumor_dataset/no/17 no.jpg  
  inflating: brain_tumor_dataset/no/18 no.jpg  
  inflating: brain_tumor_dataset/no/19 no.jpg  
  inflating: brain_tumor_dataset/no/2 no.jpeg  
  inflating: brain_tumor_dataset/no/20 no.jpg  
  inflating: brain_tumor_dataset/no/21 no.jpg  
  inflating: brain_tumor_dataset/no/22 no.jpg  
  inflating: brain_tumor_dataset/no/23 no.jpg  
  inflating: brain_tumor

In [ ]:
import os

os.listdir()


['.config',
 'yes',
 'brain-mri-images-for-brain-tumor-detection.zip',
 'test.jpg',
 'no',
 'kaggle.json',
 'brain_tumor_dataset',
 'sample_data']

In [ ]:
os.listdir("brain_tumor_dataset")


['yes', 'no']

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    "brain_tumor_dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_data = datagen.flow_from_directory(
    "brain_tumor_dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)


Found 203 images belonging to 2 classes.
Found 50 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten

base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))

# Freeze base layers (so training is faster)
for layer in base_model.layers:
    layer.trainable = False

x = Flatten()(base_model.output)
x = Dense(128, activation="relu")(x)
output = Dense(2, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.5866 - loss: 10.7570 - val_accuracy: 0.7800 - val_loss: 2.2555
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.8962 - loss: 1.4291 - val_accuracy: 0.9200 - val_loss: 0.8493
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9505 - loss: 0.1967 - val_accuracy: 0.9400 - val_loss: 0.4910
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.9980 - loss: 0.0066 - val_accuracy: 0.9200 - val_loss: 0.9268
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.9950 - loss: 0.0364 - val_accuracy: 0.9400 - val_loss: 0.8133
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.9980 - loss: 0.0019 - val_accuracy: 0.9400 - val_loss: 0.7166
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 1.0000 - loss: 5.2298e-05 - val_accuracy: 0.9400 - val_loss: 0.6496
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 1.0000 - loss: 1.0617e-05 - val_accuracy: 0.9600 - val_loss: 0.6601

In [ ]:
model.save("brain_tumor_mri_model.h5")


In [ ]:
import os
os.listdir()

['.config',
 'yes',
 'brain-mri-images-for-brain-tumor-detection.zip',
 'test.jpg',
 'no',
 'kaggle.json',
 'brain_tumor_mri_model.h5',
 'brain_tumor_dataset',
 'sample_data']

In [ ]:
loss, acc = model.evaluate(val_data)
print("Validation Accuracy:", acc)


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 520ms/step - accuracy: 0.9525 - loss: 0.8053
Validation Accuracy: 0.9599999785423279


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test.jpg to test (1).jpg


In [ ]:
import os
os.listdir()


['.config',
 'test (1).jpg',
 'yes',
 'brain-mri-images-for-brain-tumor-detection.zip',
 'test.jpg',
 'no',
 'kaggle.json',
 'brain_tumor_mri_model.h5',
 'brain_tumor_dataset',
 'sample_data']

In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing import image

model = load_model("brain_tumor_mri_model.h5")

img = image.load_img("test.jpg", target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)

if pred[0][0] > pred[0][1]:
    print("🟢 Normal Brain")
    print("Confidence:", pred[0][0])
else:
    print("🔴 Tumor Detected")
    print("Confidence:", pred[0][1])



1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
🟢 Normal Brain
Confidence: 0.9059105


In [ ]:
print(pred)


[[0.9059105  0.09408951]]
